<a href="https://colab.research.google.com/github/tarunpahade18/ml-notebooks/blob/main/QuantizedQwen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!apt-get update -qq
!apt-get install -y -qq cmake build-essential

!git clone --depth 1 https://github.com/ggml-org/llama.cpp

%cd /content/llama.cpp

!cmake -B build -DGGML_CUDA=OFF
!cmake --build build --config Release -j2

print("llama.cpp ready!")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Cloning into 'llama.cpp'...
remote: Enumerating objects: 4412, done.
remote: Counting objects: 100% (4412/4412), done.
remote: Compressing objects: 100% (3656/3656), done.
remote: Total 4412 (delta 811), reused 2915 (delta 671), pack-reused 0 (from 0)
Receiving objects: 100% (4412/4412), 37.54 MiB | 14.96 MiB/s, done.
Resolving deltas: 100% (811/811), done.
Updating files: 100% (4043/4043), done.
/content/llama.cpp
-- The C compiler identification is GNU 13.3.0
-- The CXX compiler identification is GNU 13.3.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working

In [3]:
!pip install -q -r requirements.txt
!pip install -q huggingface_hub

from huggingface_hub import snapshot_download

model_path = snapshot_download(
    repo_id="Qwen/Qwen3-0.6B",
    local_dir="/content/qwen3"
)

print("Model downloaded:", model_path)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 2.3.3 which is incompatible.
google-cloud-datastore 2.27.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 4.25.9 which is incompatible.
proto-plus 1.29.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 4.25.9 which is incompatible.
google-cloud-functions 1.25.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 4.25.9 which is incompatible.
google-cloud-firestore 2.34.1 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 4.25.9 which is incompatible.
google-cloud-storage-control 1.17.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 4.25.9 which is incompatible.
google-cloud-bigquery-co

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

merges.txt: 0.00B [00:00, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

Model downloaded: /content/qwen3


In [4]:
!pip install -q -r requirements.txt
!pip install -q huggingface_hub

from huggingface_hub import snapshot_download

model_path = snapshot_download(
    repo_id="Qwen/Qwen3-0.6B",
    local_dir="/content/qwen3"
)

print("Model downloaded:", model_path)

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Model downloaded: /content/qwen3


In [5]:
%cd /content/llama.cpp

!python convert_hf_to_gguf.py \
    /content/qwen3 \
    --outfile /content/qwen3-f16.gguf \
    --outtype f16

/content/llama.cpp
INFO:hf-to-gguf:Loading model: qwen3
INFO:hf-to-gguf:Model architecture: Qwen3ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,             torch.bfloat16 --> F16, shape = {1024, 151936}
INFO:hf-to-gguf:token_embd.weight,         torch.bfloat16 --> F16, shape = {1024, 151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.bfloat16 --> F32, shape = {1024}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.bfloat16 --> F16, shape = {3072, 1024}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.bfloat16 --> F16, shape = {1024, 3072}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.bfloat16 --> F16, shape = {1024, 3072}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.bfloat16 --> F32, shape = {1024}
INFO:hf-to-gguf:blk.0.attn_k_norm.weight,  torch.bfloat16 --> F32, shape = {128}
INFO:hf-to-gguf:blk.0.attn_k.weight,

In [6]:
!./build/bin/llama-quantize \
    /content/qwen3-f16.gguf \
    /content/qwen3-200mb.gguf \
    Q1_0

version: 0.8.0-dev (build 1, commit 28b812d)
built with GNU 13.3.0 for Linux x86_64
llama_quantize: quantizing '/content/qwen3-f16.gguf' to '/content/qwen3-200mb.gguf' as Q1_0
llama_model_loader: loaded meta data with 39 key-value pairs and 311 tensors from /content/qwen3-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen3
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 20
llama_model_loader: - kv   3:                     general.sampling.top_p f32              = 0.950000
llama_model_loader: - kv   4:                      general.sampling.temp f32              = 0.600000
llama_model_loader: - kv   5:                               general.name str              = Qwen3
ll

In [7]:
!./build/bin/llama-quantize \
    /content/qwen3-f16.gguf \
    /content/qwen3-q2.gguf \
    Q2_K

version: 0.8.0-dev (build 1, commit 28b812d)
built with GNU 13.3.0 for Linux x86_64
llama_quantize: quantizing '/content/qwen3-f16.gguf' to '/content/qwen3-q2.gguf' as Q2_K
llama_model_loader: loaded meta data with 39 key-value pairs and 311 tensors from /content/qwen3-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen3
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 20
llama_model_loader: - kv   3:                     general.sampling.top_p f32              = 0.950000
llama_model_loader: - kv   4:                      general.sampling.temp f32              = 0.600000
llama_model_loader: - kv   5:                               general.name str              = Qwen3
llama

In [8]:
import os

models = [
    "/content/qwen3-f16.gguf",
    "/content/qwen3-200mb.gguf",
    "/content/qwen3-q2.gguf"
]

for model in models:
    size = os.path.getsize(model) / (1024 * 1024)
    print(f"{os.path.basename(model)}: {size:.2f} MB")

qwen3-f16.gguf: 1439.43 MB
qwen3-200mb.gguf: 207.57 MB
qwen3-q2.gguf: 331.20 MB


In [9]:
import os

models = [
    "/content/qwen3-f16.gguf",
    "/content/qwen3-200mb.gguf",
    "/content/qwen3-q2.gguf"
]

for model in models:
    size = os.path.getsize(model) / (1024 * 1024)
    print(f"{os.path.basename(model)}: {size:.2f} MB")

qwen3-f16.gguf: 1439.43 MB
qwen3-200mb.gguf: 207.57 MB
qwen3-q2.gguf: 331.20 MB


In [ ]:
!./build/bin/llama-cli \
    -m /content/qwen3-200mb.gguf \
    -p "Explain artificial intelligence in simple terms." \
    -n 100 \
    -c 2048



Loading model... |-\|/-\|/-\ 

▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b1-28b812d
model      : /content/qwen3-200mb.gguf
ftype      : Q1_0
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read <file>        add a text file
  /glob <pattern>     add text files using globbing pattern



> Explain artificial intelligence in simple terms.
)))


hydrationPORPOR.'&































 OpsOpsOpsenezOps































































atura































 }));

geberaturawideigenampireops-*OpsaturaشروaturaOpsampireatura圬OpsestOpsOps圬awayoperarcyampireOpsawayaway,exportsشرو={({={({Ops

In [ ]:
from google.colab import files

files.download("/content/qwen3-200mb.gguf")